# Projet MLSF – Partie I : solveur SAT pour Meowdoku
**Pablo Bosch, Cécile Chevrière**

Les programmes sont regroupés dans `meowdoku.py` (grilles, encodage CNF, DIMACS, vérification) et `generateur.py` (génération de grilles). Ce notebook les exécute dans l'ordre des questions ; la démarche et l'analyse des résultats sont dans le rapport.

In [ ]:
import itertools, os, random
from meowdoku import (lire_grille, ecrire_grille, afficher, grille_vers_cnf, ecrire_dimacs,
                      convertir, var, satisfait, chats_depuis_valuation, verifier_solution)
from generateur import generer_grille, generer_fichier

os.makedirs("grilles", exist_ok=True)
os.makedirs("cnf", exist_ok=True)

## Question 1 – Modélisation de Meowdoku

### 1.1 Conversion grille → DIMACS
Grille de la figure 1 de l'énoncé : lecture, conversion, aperçu du fichier produit.

In [ ]:
grille = lire_grille("grilles/grille_enonce.txt")
afficher(grille)
nb_vars, nb_clauses = convertir("grilles/grille_enonce.txt", "cnf/grille_enonce.cnf")
print(f"\n{nb_vars} variables, {nb_clauses} clauses")
with open("cnf/grille_enonce.cnf") as f:
    print("".join(f.readlines()[:10]) + "...")

### 1.2 Validation de l'encodage
**a.** La solution de la figure 2 est licite et satisfait la CNF.

In [ ]:
solution_fig2 = {(3, 2), (2, 5), (1, 3), (5, 6), (4, 4), (6, 1)}
_, clauses = grille_vers_cnf(grille)
print("règles violées :", verifier_solution(grille, solution_fig2))
print("CNF satisfaite :", satisfait(clauses, {var(i, j, grille.M) for i, j in solution_fig2}))

**b.** Test exhaustif sur des grilles 4×4 : pour chacune des 2¹⁶ valuations, on compare « la valuation satisfait la CNF » et « la configuration de chats associée est licite ».

In [ ]:
def comparer(grille, clauses, vraies):
    return satisfait(clauses, vraies) == (not verifier_solution(grille, chats_depuis_valuation(vraies, grille.M)))

for graine in range(5):
    g, _ = generer_grille(4, graine)
    nv, cl = grille_vers_cnf(g)
    modeles, accord = 0, True
    for bits in range(2 ** nv):
        vraies = {v for v in range(1, nv + 1) if bits >> (v - 1) & 1}
        accord &= comparer(g, cl, vraies)
        modeles += satisfait(cl, vraies)
    print(f"grille 4x4 n°{graine} : accord sur les 2^{nv} valuations = {accord}, modèles = {modeles}")

**c.** Grilles plus grandes (2^NM valuations hors de portée) : toutes les permutations (un chat par ligne et par colonne) et 20 000 valuations aléatoires.

In [ ]:
rng = random.Random(0)
for N, graine in [(6, 0), (7, 1)]:
    g, _ = generer_grille(N, graine)
    nv, cl = grille_vers_cnf(g)
    perms = [{var(i + 1, p[i] + 1, N) for i in range(N)} for p in itertools.permutations(range(N))]
    aleatoires = [{v for v in range(1, nv + 1) if rng.random() < rng.choice([0.05, 0.15, 0.3])}
                  for _ in range(20000)]
    ok = all(comparer(g, cl, s) for s in perms + aleatoires)
    print(f"{N}x{N} : {len(perms)} permutations + 20000 valuations aléatoires, accord = {ok}, "
          f"permutations solutions = {sum(satisfait(cl, s) for s in perms)}")

### 1.3 Générateur de grilles (N = M = K)
Exemple de grille 8×8 générée, avec la solution construite (`*`).

In [ ]:
g, chats = generer_fichier(8, 2026, "grilles/exemple_8x8.txt")
afficher(g, chats)
print("\nrègles violées par la solution construite :", verifier_solution(g, chats))

Pas de configuration licite pour N = 2 et N = 3 :

In [ ]:
for N in (2, 3):
    try:
        generer_grille(N)
    except ValueError as e:
        print(e)

### 1.4 Jeu de grilles générées et fichiers DIMACS
Grilles de taille 4 à 20 (3 graines par taille), vérification de la solution construite et conversion en DIMACS. Ces instances serviront aux questions suivantes.

In [ ]:
print(f"{'N':>3} {'variables':>10} {'clauses':>9}  solutions construites licites")
for N in range(4, 21):
    tailles, licites = [], True
    for graine in range(3):
        nom = f"meow_{N:02d}_{graine}"
        g, chats = generer_fichier(N, graine, f"grilles/{nom}.txt")
        licites &= not verifier_solution(g, chats)
        tailles.append(convertir(f"grilles/{nom}.txt", f"cnf/{nom}.cnf"))
    nv = tailles[0][0]
    nc = sum(t[1] for t in tailles) // 3
    print(f"{N:>3} {nv:>10} {nc:>9}  {licites}")